In [2]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import warnings
import logging
from torch_brain.utils import seed_everything

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

warnings.filterwarnings('ignore')
logging.disable(logging.WARNING)

def move_to_gpu(data, device):
    """
    Recursively moves tensors (or collections of tensors) to the given device.
    """
    if isinstance(data, torch.Tensor):
        return data.to(device)
    elif isinstance(data, dict):
        return {k: move_to_gpu(v, device) for k, v in data.items()}
    elif isinstance(data, list):
        return [move_to_gpu(elem, device) for elem in data]
    else:
        return data

def r2_score(y_pred, y_true):
    # Compute total sum of squares (variance of the true values)
    y_true_mean = torch.mean(y_true, dim=0, keepdim=True)
    ss_total = torch.sum((y_true - y_true_mean) ** 2)

    # Compute residual sum of squares
    ss_res = torch.sum((y_true - y_pred) ** 2)

    # Compute R^2
    r2 = 1 - ss_res / ss_total

    return r2

def compute_r2(dataloader, model):
    # Compute R2 score over the entire dataset
    total_target = []
    total_pred = []
    for batch in dataloader:
        batch = move_to_gpu(batch, device)
        print('Passing to model')
        pred = model(**batch["model_inputs"])
        target = batch["target_values"]
        
        pred = pred.cpu()
        target = target.cpu()

        # Store target and pred for visualization
        mask = torch.ones_like(target, dtype=torch.bool)
        if "output_mask" in batch["model_inputs"]:
            mask = batch["model_inputs"]["output_mask"]
        mask = mask.cpu()
        total_target.append(target[mask])
        total_pred.append(pred[mask])
        print('Ended model ')
        
    # Concatenate all batch outputs
    total_target = torch.cat(total_target)
    total_pred = torch.cat(total_pred)

    # Compute the R2 score
    r2 = r2_score(total_pred.flatten(), total_target.flatten())

    return r2.item(), total_target, total_pred

def print_model(model: torch.nn.Module):
    """
    Prints a summary of the model architecture and parameter count.
    """
    model_str = str(model).split('\n')
    print("\nModel:")
    print('\n'.join(model_str[:5]))
    print("...")
    print('\n'.join(model_str[-min(5, len(model_str)):]))
    num_params = sum(p.numel() for p in model.parameters())
    if num_params > 1e9:
        param_str = f"{num_params/1e9:.1f}G"
    elif num_params > 1e6:
        param_str = f"{num_params/1e6:.1f}M"
    else:
        param_str = f"{num_params/1e3:.1f}K"
    print(f"\nNumber of parameters: {param_str}\n")

def plot_training_curves(r2_log, loss_log):
    """
    Plots the training curves: training loss and validation R2 score.
    """
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(np.linspace(0, len(loss_log), len(loss_log)), loss_log)
    plt.title("Training Loss")
    plt.xlabel("Training Steps")
    plt.ylabel("MSE Loss")
    plt.grid()
    plt.subplot(1, 2, 2)
    plt.plot(r2_log)
    plt.title("Validation R2")
    plt.xlabel("Epochs")
    plt.ylabel("R2 Score")
    plt.grid()
    plt.tight_layout()
    plt.show()

def generate_sinusoidal_position_embs(num_timesteps, dim):
    position = torch.arange(num_timesteps).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, dim, 2) * (-np.log(10000.0) / dim))
    pe = torch.empty(num_timesteps, dim)
    pe[:, 0:dim // 2] = torch.sin(position * div_term)
    pe[:, dim//2:] = torch.cos(position * div_term)
    return pe

def load_pretrained(ckpt_path, model):
    print("Loading pretrained model...")
    checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    # poyo is pretrained using lightning, so model weights are prefixed with "model."
    state_dict = {k.replace("model.", ""): v for k, v in checkpoint["state_dict"].items()}
    model.load_state_dict(state_dict)
    print("Done!")
    return model

def reinit_vocab(emb_module, vocab):
    emb_module.extend_vocab(vocab)
    emb_module.subset_vocab(vocab)

def get_dataset_config(brainset, sessions):
    brainset_norms = {
        "perich_miller_population_2018": {
            "mean": 0.0,
            "std": 20.0
        },
        "rat_hippocampus": {
            "mean": 0.0,
            "std": 1.0
        }
    }
    
    config = f"""
    - selection:
      - brainset: {brainset}
        sessions:"""
    if type(sessions) is not list:
        sessions = [sessions]
    for session in sessions:
        config += f"""
          - {session}"""
    rid = "cursor_velocity_2d" if brainset == "perich_miller_population_2018" else "linear_maze_pos"
    config += f"""
      config:
        readout:
          readout_id: {rid}
          normalize_mean: {brainset_norms[brainset]["mean"]}
          normalize_std: {brainset_norms[brainset]["std"]}
          metrics:
            - metric:
                _target_: torchmetrics.R2Score
    """

    config = OmegaConf.create(config)

    return config


In [7]:
from torch_brain.data import Dataset, collate, chain
from torch_brain.data.sampler import RandomFixedWindowSampler, SequentialFixedWindowSampler
from torch.utils.data import DataLoader
from torch_brain.data.dataset import DatasetIndex
import math 

class BalancedSessionBatchSampler(torch.utils.data.Sampler):
    def __init__(
        self,
        sampling_intervals,
        window_length,
        batch_size,
        generator=None,
        drop_short=True,
        drop_last=True,
        oversample_short_sessions=True,
    ):
        self.sampling_intervals = sampling_intervals
        self.window_length = window_length
        self.batch_size = batch_size
        self.generator = generator
        self.drop_short = drop_short
        self.drop_last = drop_last
        self.oversample_short_sessions = oversample_short_sessions
        self.session_names = list(sampling_intervals.keys())
        assert batch_size % len(self.session_names) == 0
        self.samples_per_session = batch_size // len(self.session_names)

    def _sample_session_windows(self, session_name, intervals):
        indices = []

        for start, end in zip(intervals.start, intervals.end):
            interval_length = end - start

            if interval_length < self.window_length:
                if self.drop_short:
                    continue
                raise ValueError(
                    f"Interval {(start, end)} is too short to sample from."
                )

            left_offset = (
                torch.rand(1, generator=self.generator).item() * self.window_length
            )
            
            indices_ = [
                DatasetIndex(session_name, t.item(), (t + self.window_length).item())
                for t in torch.arange(
                    start + left_offset,
                    end,
                    self.window_length,
                    dtype=torch.float64,
                )
                if t + self.window_length <= end
            ]
            
            indices.extend(indices_)

        return indices

    def __iter__(self):
        per_session_indices = {}

        for session_name, intervals in self.sampling_intervals.items():
            indices = self._sample_session_windows(session_name, intervals)

            if len(indices) == 0:
                raise ValueError(f"No valid samples for session {session_name}")

            perm = torch.randperm(len(indices), generator=self.generator)
            per_session_indices[session_name] = [indices[i] for i in perm]

        if self.oversample_short_sessions:
            max_len = max(len(v) for v in per_session_indices.values())
            num_batches = math.ceil(max_len / self.samples_per_session)
        else:
            min_len = min(len(v) for v in per_session_indices.values())
            num_batches = min_len // self.samples_per_session

        for _ in range(num_batches):
            batch = []

            for session_name in self.session_names:
                indices = per_session_indices[session_name]

                for _ in range(self.samples_per_session):
                    if len(indices) == 0:
                        if not self.oversample_short_sessions:
                            break

                        # resample this session
                        refreshed = self._sample_session_windows(
                            session_name,
                            self.sampling_intervals[session_name],
                        )
                        perm = torch.randperm(len(refreshed), generator=self.generator)
                        indices.extend([refreshed[i] for i in perm])

                    batch.append(indices.pop())

            if len(batch) == self.batch_size or not self.drop_last:
                perm = torch.randperm(len(batch), generator=self.generator)
                yield [batch[i] for i in perm]

    def __len__(self):
        per_session_counts = {}

        for session_name, intervals in self.sampling_intervals.items():
            count = 0
            for start, end in zip(intervals.start, intervals.end):
                interval_length = end - start
                if interval_length >= self.window_length:
                    count += math.floor(interval_length / self.window_length)

            per_session_counts[session_name] = count

        if self.oversample_short_sessions:
            max_len = max(per_session_counts.values())
            return math.ceil(max_len / self.samples_per_session)
        else:
            min_len = min(per_session_counts.values())
            return min_len // self.samples_per_session

def get_train_val_loaders(recording_id=None, cfg=None, batch_size=32, seed=0):
    """Sets up train and validation Datasets, Samplers, and DataLoaders
    """
    root = "D:/Pose/Neuro Code/data/NoveltySessInfoMatFiles/linear_processed/hippo_processed"
    # -- Train --
    train_dataset = Dataset(
        root=root,                # root directory where .h5 files are found
        recording_id=recording_id,  # you either specify a single recording ID
        config=cfg,                 # or a config for multi-session training / more complex configs
        split="train",
    )
    # We use a random sampler to improve generalization during training
    train_sampling_intervals = train_dataset.get_sampling_intervals()
    print(train_sampling_intervals)
    train_sampler = RandomFixedWindowSampler(
        sampling_intervals=train_sampling_intervals,
        window_length=1.0,          # context window of samples
        generator=torch.Generator().manual_seed(seed),
    )
    train_batch_sampler = BalancedSessionBatchSampler(
        sampling_intervals=train_sampling_intervals,
        window_length=1.0,
        batch_size=64,
        generator=torch.Generator().manual_seed(seed),
        oversample_short_sessions=True,
    )

    # Finally combine them in a dataloader
    train_loader = DataLoader(
        dataset=train_dataset,      # dataset
        # sampler=train_sampler,      # sampler
        batch_sampler=train_batch_sampler,
        # batch_size=batch_size,      # num of samples per batch
        collate_fn=collate,         # the collator
        # num_workers=1,              # data sample processing (slicing, transforms, tokenization) happens in parallel; this sets the amount of that parallelization
        # pin_memory=True,
    )

    # -- Validation --
    val_dataset = Dataset(
        root=root,
        recording_id=recording_id,
        config=cfg,
        split="valid",
    )
    # For validation we don't randomize samples for reproducibility
    val_sampling_intervals = val_dataset.get_sampling_intervals()
    val_sampler = SequentialFixedWindowSampler(
        sampling_intervals=val_sampling_intervals,
        window_length=1.0,
    )
    # Combine them in a dataloader
    val_loader = DataLoader(
        dataset=val_dataset,
        sampler=val_sampler,
        batch_size=batch_size,
        collate_fn=collate,
        # num_workers=1,
        # pin_memory=True,
    )

    train_dataset.disable_data_leakage_check()
    val_dataset.disable_data_leakage_check()

    return train_dataset, train_loader, val_dataset, val_loader



In [8]:

import torch.nn.functional as F
from tqdm import trange 

def train(model, optimizer, train_loader, val_loader, num_epochs=50, store_embs=False):
    # We'll store some intermediate outputs for visualization
    train_outputs = {
        'n_epochs': num_epochs,
        'unit_emb': [],
        'session_emb': [],
        'output_pred': [],
        'output_gt': [],
    }

    r2_log = []
    loss_log = []
    before_gip = []
    after_gip = []

    # Training loop
    for epoch in trange(num_epochs):
        # Compute R² score on validation set
        # print('evaling')
        r2, target, pred = compute_r2(val_loader, model)
        # print('completed eval')
        r2_log.append(r2)

        # Training steps
        for batch in train_loader:
            batch = move_to_gpu(batch, device)
            before_gip.append(gip(batch, model, optimizer).item())
            loss = training_step(batch, model, optimizer)
            loss_log.append(loss.item())
            after_gip.append(gip(batch, model, optimizer).item())
            
        print(f"\rEpoch {epoch+1}/{num_epochs} | Val R2 = {r2:.3f} | Loss = {loss.item():.3f}", end="")

        # Store intermediate outputs
        if store_embs:
            train_outputs['unit_emb'].append(model.unit_emb.weight[1:].detach().cpu().numpy())
            train_outputs['session_emb'].append(model.session_emb.weight[1:].detach().cpu().numpy())
        train_outputs['output_gt'].append(target.detach().cpu().numpy())
        train_outputs['output_pred'].append(pred.detach().cpu().numpy())

    # Compute final R² score
    r2, _, _ = compute_r2(val_loader, model)
    r2_log.append(r2)
    print(f"\nDone! Final validation R2 = {r2:.3f}")

    return r2_log, loss_log, train_outputs, before_gip, after_gip

from collections import defaultdict
import torch


def training_step(batch, model, optimizer):
    optimizer.zero_grad()                  # Step 0. Clear old gradients
    pred = model(**batch["model_inputs"])  # Step 1. Do forward pass
    target = batch["target_values"]
    loss = F.mse_loss(pred, target)        # Step 2. Compute loss
    loss.backward()                        # Step 3. Backward pass
    optimizer.step()                       # Step 4. Update model params
    return loss

def take_indices(x, idx):
    if torch.is_tensor(x):
        return x[idx]
    if isinstance(x, dict):
        return {k: take_indices(v, idx) for k, v in x.items()}
    if isinstance(x, list):
        return [x[i] for i in idx.tolist()]
    return x

def gip(batch, model, optimizer): 
    session_ids = batch["session_id"]
    grads = {}
    
    domain_to_indices = defaultdict(list)
    for i, domain in enumerate(session_ids):
        domain_to_indices[domain].append(i)
        grads[domain] = []

    for domain, indices in domain_to_indices.items():
        optimizer.zero_grad()
        idx = torch.tensor(indices, device=batch["target_values"].device)

        model_inputs = take_indices(batch["model_inputs"], idx)
        target = take_indices(batch["target_values"], idx)
        pred = model(**model_inputs)

        loss = F.mse_loss(pred, target) 
        loss.backward()
            
        for name, param in model.named_parameters():
            if param.grad is not None:
                if name not in ['unit_emb.weight', 'session_emb.weight', 'token_type_emb.weight', 'latent_emb.weight']:
                    # grads[domain][name] = param.grad.clone()
                    grads[domain].append(param.grad.clone().flatten())

        optimizer.zero_grad()

    grad_mat = []
    for domain, domain_grads in grads.items():
        grads[domain] = torch.cat(grads[domain], dim=0)
        grad_mat.append(grads[domain])
    
    grad_mat = torch.stack(grad_mat, dim=0)  # [K, N]
    X = F.normalize(grad_mat, dim=1)
    sim = X @ X.T 
    avg_sim = torch.tril(sim, diagonal=-1).mean()

    return avg_sim

"""
session_ids = batch["session_id"]  # list of N strings

domain_to_indices = defaultdict(list)
for i, domain in enumerate(session_ids):
    domain_to_indices[domain].append(i)

for domain, indices in domain_to_indices.items():
    idx = torch.tensor(indices, device=batch["target_values"].device)

    model_inputs = take_indices(batch["model_inputs"], idx)
    target = take_indices(batch["target_values"], idx)

    pred = model(**model_inputs)

    loss = criterion(pred, target)
    loss.backward()
"""

'\nsession_ids = batch["session_id"]  # list of N strings\n\ndomain_to_indices = defaultdict(list)\nfor i, domain in enumerate(session_ids):\n    domain_to_indices[domain].append(i)\n\nfor domain, indices in domain_to_indices.items():\n    idx = torch.tensor(indices, device=batch["target_values"].device)\n\n    model_inputs = take_indices(batch["model_inputs"], idx)\n    target = take_indices(batch["target_values"], idx)\n\n    pred = model(**model_inputs)\n\n    loss = criterion(pred, target)\n    loss.backward()\n'

In [9]:
import torch 
import torch.nn.functional as F 
input1 = torch.randn(100, 128)
input2 = torch.randn(100, 128)
output = F.cosine_similarity(input1, input2)
print(output.shape)

torch.Size([100])


In [10]:
seed_everything(0)

# 1. Setup datasets and dataloader
# For a model like POYO, which was built for multi-session training, the way to
# instantiate a dataset is just slightly more involved than what we have used
# so far. We have abstracted that difference in a utility function `get_dataset_config`
train_dataset, train_loader, val_dataset, val_loader = get_train_val_loaders(
    cfg=get_dataset_config("rat_hippocampus",
        ["achilles_10252013_sessinfo",
         "buddy_06272013_sessinfo",
         "cicero_09012014_sessinfo", # 
         "gatsby_08022013_sessinfo"
         ]),
    batch_size=64, # 64,
)

for i, batch in enumerate(train_loader):
    print(batch.keys())
    print(i, batch['model_inputs']['input_timestamps'].shape) 
    # print(batch['model_inputs']['output_timestamps'][:, 0:3], batch['model_inputs']['output_timestamps'][0, -3:])
    # print()
    break

sampling_intervals_dict {'rat_hippocampus/achilles_10252013_sessinfo': LazyInterval(
  end=<HDF5 dataset "end": shape (1,), type "<f8">,
  start=<HDF5 dataset "start": shape (1,), type "<f8">
), 'rat_hippocampus/buddy_06272013_sessinfo': LazyInterval(
  end=<HDF5 dataset "end": shape (1,), type "<f8">,
  start=<HDF5 dataset "start": shape (1,), type "<f8">
), 'rat_hippocampus/cicero_09012014_sessinfo': LazyInterval(
  end=<HDF5 dataset "end": shape (1,), type "<f8">,
  start=<HDF5 dataset "start": shape (1,), type "<f8">
), 'rat_hippocampus/gatsby_08022013_sessinfo': LazyInterval(
  end=<HDF5 dataset "end": shape (1,), type "<f8">,
  start=<HDF5 dataset "start": shape (1,), type "<f8">
)}
{'rat_hippocampus/achilles_10252013_sessinfo': LazyInterval(
  end=<HDF5 dataset "end": shape (1,), type "<f8">,
  start=<HDF5 dataset "start": shape (1,), type "<f8">
), 'rat_hippocampus/buddy_06272013_sessinfo': LazyInterval(
  end=<HDF5 dataset "end": shape (1,), type "<f8">,
  start=<HDF5 dataset 

TypeError: default_collate: batch must contain tensors, numpy arrays, numbers, dicts or lists; found <class 'temporaldata.temporaldata.Data'>

In [6]:

seed_everything(0)

# 1. Setup datasets and dataloader
# For a model like POYO, which was built for multi-session training, the way to
# instantiate a dataset is just slightly more involved than what we have used
# so far. We have abstracted that difference in a utility function `get_dataset_config`
train_dataset, train_loader, val_dataset, val_loader = get_train_val_loaders(
    cfg=get_dataset_config("rat_hippocampus",
        ["achilles_10252013_sessinfo",
         "buddy_06272013_sessinfo",
         "cicero_09012014_sessinfo", # 
         "gatsby_08022013_sessinfo"
         ]),
    batch_size=64, # 64,
)

# 2. Instantiate the model. The model implementation is provided by torch_brain
from torch_brain.models import POYO
from torch_brain.registry import MODALITY_REGISTRY
poyo_model = POYO(
    sequence_length=1.0,                                    # Context length of the model
    readout_spec=MODALITY_REGISTRY['linear_maze_pos'],   # POYO allows for multiple readout modalities; this is how we choose
    #
    latent_step=1.0 / 8,                                    # Timestep of the learned latent grid
    num_latents_per_step=16,                                # Number of unique learned latents  per timestep
    #
    dim=64,                                                 # Hidden dimension of the model
    depth=6,                                                # Number of transformer layers
    #
    dim_head=64,                                            # Dimension of each attention head
    cross_heads=2,                                          # Num. of heads in cross-attention blocks
    self_heads=8,                                           # Num. of heads in self attention blocks
).to(device)

# 2.5: Extra step: populate the Unit and Session Embedding Vocabularies
poyo_model.unit_emb.initialize_vocab(train_dataset.get_unit_ids())
poyo_model.session_emb.initialize_vocab(train_dataset.get_session_ids())

# 3. Connect tokenizers to Datasets
train_dataset.transform = poyo_model.tokenize
val_dataset.transform = poyo_model.tokenize

poyo_model

sampling_intervals_dict {'rat_hippocampus/achilles_10252013_sessinfo': LazyInterval(
  end=<HDF5 dataset "end": shape (1,), type "<f8">,
  start=<HDF5 dataset "start": shape (1,), type "<f8">
), 'rat_hippocampus/buddy_06272013_sessinfo': LazyInterval(
  end=<HDF5 dataset "end": shape (1,), type "<f8">,
  start=<HDF5 dataset "start": shape (1,), type "<f8">
), 'rat_hippocampus/cicero_09012014_sessinfo': LazyInterval(
  end=<HDF5 dataset "end": shape (1,), type "<f8">,
  start=<HDF5 dataset "start": shape (1,), type "<f8">
), 'rat_hippocampus/gatsby_08022013_sessinfo': LazyInterval(
  end=<HDF5 dataset "end": shape (1,), type "<f8">,
  start=<HDF5 dataset "start": shape (1,), type "<f8">
)}
{'rat_hippocampus/achilles_10252013_sessinfo': LazyInterval(
  end=<HDF5 dataset "end": shape (1,), type "<f8">,
  start=<HDF5 dataset "start": shape (1,), type "<f8">
), 'rat_hippocampus/buddy_06272013_sessinfo': LazyInterval(
  end=<HDF5 dataset "end": shape (1,), type "<f8">,
  start=<HDF5 dataset 

POYO(
  (unit_emb): InfiniteVocabEmbedding(embedding_dim=64, num_embeddings=290)
  (session_emb): InfiniteVocabEmbedding(embedding_dim=64, num_embeddings=5)
  (token_type_emb): Embedding(4, 64)
  (latent_emb): Embedding(16, 64)
  (rotary_emb): RotaryTimeEmbedding()
  (dropout): Dropout(p=0.4, inplace=False)
  (enc_atn): RotaryCrossAttention(
    (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    (norm_context): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    (to_q): Linear(in_features=64, out_features=128, bias=False)
    (to_kv): Linear(in_features=64, out_features=256, bias=False)
    (to_out): Linear(in_features=128, out_features=64, bias=True)
  )
  (enc_ffn): Sequential(
    (0): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    (1): FeedForward(
      (net): Sequential(
        (0): Linear(in_features=64, out_features=512, bias=True)
        (1): GEGLU()
        (2): Dropout(p=0.2, inplace=False)
        (3): Linear(in_features=256, out_features=64, 

In [11]:
# for i, batch in enumerate(train_loader):
#     print(batch['model_inputs'].keys())
#     print(batch.keys())
#     print(len(batch['session_id']))
#     break
    

In [12]:
# counts = {
#     "rat_hippo/cicero_09012014_sessinfo": 0,
#     "rat_hippo/achilles_10252013_sessinfo": 0,
#     "rat_hippo/gatsby_08022013_sessinfo": 0,
#     "rat_hippo/buddy_06272013_sessinfo": 0,
# }

# for i, batch in enumerate(train_loader):
#     print(batch['model_inputs']['input_timestamps'].shape)
#     your_list = batch['session_id']
#     curr_counts = {
#         "rat_hippo/cicero_09012014_sessinfo": 0,
#         "rat_hippo/achilles_10252013_sessinfo": 0,
#         "rat_hippo/gatsby_08022013_sessinfo": 0,
#         "rat_hippo/buddy_06272013_sessinfo": 0,
#     }
#     for s in your_list:   # loop over your strings
#         if s in counts:
#             counts[s] += 1
#             curr_counts[s] += 1

#     print(curr_counts)

# print(counts)

In [16]:
for name, _ in poyo_model.named_parameters():
    if name not in ['unit_emb.weight', 'session_emb.weight', 'token_type_emb.weight', 'latent_emb.weight']:
        print(name)

enc_atn.norm.weight
enc_atn.norm.bias
enc_atn.norm_context.weight
enc_atn.norm_context.bias
enc_atn.to_q.weight
enc_atn.to_kv.weight
enc_atn.to_out.weight
enc_atn.to_out.bias
enc_ffn.0.weight
enc_ffn.0.bias
enc_ffn.1.net.0.weight
enc_ffn.1.net.0.bias
enc_ffn.1.net.3.weight
enc_ffn.1.net.3.bias
proc_layers.0.0.norm.weight
proc_layers.0.0.norm.bias
proc_layers.0.0.to_qkv.weight
proc_layers.0.0.to_out.weight
proc_layers.0.0.to_out.bias
proc_layers.0.1.0.weight
proc_layers.0.1.0.bias
proc_layers.0.1.1.net.0.weight
proc_layers.0.1.1.net.0.bias
proc_layers.0.1.1.net.3.weight
proc_layers.0.1.1.net.3.bias
proc_layers.1.0.norm.weight
proc_layers.1.0.norm.bias
proc_layers.1.0.to_qkv.weight
proc_layers.1.0.to_out.weight
proc_layers.1.0.to_out.bias
proc_layers.1.1.0.weight
proc_layers.1.1.0.bias
proc_layers.1.1.1.net.0.weight
proc_layers.1.1.1.net.0.bias
proc_layers.1.1.1.net.3.weight
proc_layers.1.1.1.net.3.bias
proc_layers.2.0.norm.weight
proc_layers.2.0.norm.bias
proc_layers.2.0.to_qkv.weight
p

In [1]:
"""
# unit_emb.weight
# session_emb.weight
# token_type_emb.weight
# latent_emb.weight
enc_atn.norm.weight
enc_atn.norm.bias
enc_atn.norm_context.weight
enc_atn.norm_context.bias
enc_atn.to_q.weight
enc_atn.to_kv.weight
enc_atn.to_out.weight
enc_atn.to_out.bias
enc_ffn.0.weight
enc_ffn.0.bias
enc_ffn.1.net.0.weight
enc_ffn.1.net.0.bias
enc_ffn.1.net.3.weight
enc_ffn.1.net.3.bias
proc_layers.0.0.norm.weight
proc_layers.0.0.norm.bias
proc_layers.0.0.to_qkv.weight
proc_layers.0.0.to_out.weight
proc_layers.0.0.to_out.bias
proc_layers.0.1.0.weight
proc_layers.0.1.0.bias
proc_layers.0.1.1.net.0.weight
proc_layers.0.1.1.net.0.bias
proc_layers.0.1.1.net.3.weight
proc_layers.0.1.1.net.3.bias
proc_layers.1.0.norm.weight
proc_layers.1.0.norm.bias
proc_layers.1.0.to_qkv.weight
proc_layers.1.0.to_out.weight
proc_layers.1.0.to_out.bias
proc_layers.1.1.0.weight
proc_layers.1.1.0.bias
proc_layers.1.1.1.net.0.weight
proc_layers.1.1.1.net.0.bias
proc_layers.1.1.1.net.3.weight
proc_layers.1.1.1.net.3.bias
proc_layers.2.0.norm.weight
proc_layers.2.0.norm.bias
proc_layers.2.0.to_qkv.weight
proc_layers.2.0.to_out.weight
proc_layers.2.0.to_out.bias
proc_layers.2.1.0.weight
proc_layers.2.1.0.bias
proc_layers.2.1.1.net.0.weight
proc_layers.2.1.1.net.0.bias
proc_layers.2.1.1.net.3.weight
proc_layers.2.1.1.net.3.bias
proc_layers.3.0.norm.weight
proc_layers.3.0.norm.bias
proc_layers.3.0.to_qkv.weight
proc_layers.3.0.to_out.weight
proc_layers.3.0.to_out.bias
proc_layers.3.1.0.weight
proc_layers.3.1.0.bias
proc_layers.3.1.1.net.0.weight
proc_layers.3.1.1.net.0.bias
proc_layers.3.1.1.net.3.weight
proc_layers.3.1.1.net.3.bias
proc_layers.4.0.norm.weight
proc_layers.4.0.norm.bias
proc_layers.4.0.to_qkv.weight
proc_layers.4.0.to_out.weight
proc_layers.4.0.to_out.bias
proc_layers.4.1.0.weight
proc_layers.4.1.0.bias
proc_layers.4.1.1.net.0.weight
proc_layers.4.1.1.net.0.bias
proc_layers.4.1.1.net.3.weight
proc_layers.4.1.1.net.3.bias
proc_layers.5.0.norm.weight
proc_layers.5.0.norm.bias
proc_layers.5.0.to_qkv.weight
proc_layers.5.0.to_out.weight
proc_layers.5.0.to_out.bias
proc_layers.5.1.0.weight
proc_layers.5.1.0.bias
proc_layers.5.1.1.net.0.weight
proc_layers.5.1.1.net.0.bias
proc_layers.5.1.1.net.3.weight
proc_layers.5.1.1.net.3.bias
dec_atn.norm.weight
dec_atn.norm.bias
dec_atn.norm_context.weight
dec_atn.norm_context.bias
dec_atn.to_q.weight
dec_atn.to_kv.weight
dec_atn.to_out.weight
dec_atn.to_out.bias
dec_ffn.0.weight
dec_ffn.0.bias
dec_ffn.1.net.0.weight
dec_ffn.1.net.0.bias
dec_ffn.1.net.3.weight
dec_ffn.1.net.3.bias
readout.weight
readout.bias
"""


'\n# unit_emb.weight\n# session_emb.weight\n# token_type_emb.weight\n# latent_emb.weight\nenc_atn.norm.weight\nenc_atn.norm.bias\nenc_atn.norm_context.weight\nenc_atn.norm_context.bias\nenc_atn.to_q.weight\nenc_atn.to_kv.weight\nenc_atn.to_out.weight\nenc_atn.to_out.bias\nenc_ffn.0.weight\nenc_ffn.0.bias\nenc_ffn.1.net.0.weight\nenc_ffn.1.net.0.bias\nenc_ffn.1.net.3.weight\nenc_ffn.1.net.3.bias\nproc_layers.0.0.norm.weight\nproc_layers.0.0.norm.bias\nproc_layers.0.0.to_qkv.weight\nproc_layers.0.0.to_out.weight\nproc_layers.0.0.to_out.bias\nproc_layers.0.1.0.weight\nproc_layers.0.1.0.bias\nproc_layers.0.1.1.net.0.weight\nproc_layers.0.1.1.net.0.bias\nproc_layers.0.1.1.net.3.weight\nproc_layers.0.1.1.net.3.bias\nproc_layers.1.0.norm.weight\nproc_layers.1.0.norm.bias\nproc_layers.1.0.to_qkv.weight\nproc_layers.1.0.to_out.weight\nproc_layers.1.0.to_out.bias\nproc_layers.1.1.0.weight\nproc_layers.1.1.0.bias\nproc_layers.1.1.1.net.0.weight\nproc_layers.1.1.1.net.0.bias\nproc_layers.1.1.1.net

In [7]:
# 4. Setup Optimizer
optimizer = torch.optim.AdamW(poyo_model.parameters(), lr=1e-3)

# 5. Train!
poyo_r2_log, poyo_loss_log, poyo_train_outputs, before_gip, after_gip = train(
  poyo_model, optimizer, train_loader, val_loader,
  num_epochs=10, store_embs=False, # True,
)

# Plot the training loss and validation R2
plot_training_curves(poyo_r2_log, poyo_loss_log)

  0%|          | 0/10 [00:00<?, ?it/s]

Passing to model
Ended model 
Passing to model
Ended model 
Passing to model
Ended model 
Passing to model
Ended model 
Passing to model
Ended model 
Passing to model
Ended model 
Passing to model
Ended model 


  0%|          | 0/10 [02:29<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
plot_training_curves(before_gip, after_gip)

0 torch.Size([64, 512])
1 torch.Size([64, 208])
2 torch.Size([64, 208])
3 torch.Size([64, 216])
4 torch.Size([64, 208])
5 torch.Size([64, 288])
6 torch.Size([34, 280])
0 torch.Size([64, 520])
1 torch.Size([64, 528])
2 torch.Size([64, 544])
3 torch.Size([64, 528])
4 torch.Size([64, 560])
5 torch.Size([64, 608])
6 torch.Size([64, 504])
7 torch.Size([64, 528])
8 torch.Size([64, 568])
9 torch.Size([64, 480])
10 torch.Size([64, 568])
11 torch.Size([64, 544])
12 torch.Size([64, 544])
13 torch.Size([64, 552])
14 torch.Size([64, 528])
15 torch.Size([64, 568])
16 torch.Size([64, 552])
17 torch.Size([64, 512])
18 torch.Size([64, 568])
19 torch.Size([32, 520])
